In [1]:
import sys
import asyncio
import time
import os

import numpy as np

from lsst.ts import salobj

from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS

In [2]:
# This notebook is an attempt to track down the source 
# of the electrical noise which causes the horizontal banding in AuxTel images.

In [3]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [4]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [5]:
#Start classes
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.01 sec
Read 1 history items for RemoteEvent(ATHeaderService, 0, authList)
Read 100 history items for RemoteEvent(ATHeaderService, 0, heartbeat)
Read 100 history items for RemoteEvent(ATHeaderService, 0, largeFileObjectAvailable)
Read 1 history items for RemoteEvent(ATHeaderService, 0, logLevel)
Read 100 history items for RemoteEvent(ATHeaderService, 0, logMessage)
Read 1 history items for RemoteEvent(ATHeaderService, 0, simulationMode)
Read 1 history items for RemoteEvent(ATHeaderService, 0, softwareVersions)
Read 13 history items for RemoteEvent(ATHeaderService, 0, summaryState)
Read historical da

[[None, None, None, None, None, None, None], [None, None, None, None]]

In [6]:
# enable components
await atcs.enable()
await latiss.enable()

Enabling all components
Gathering settings.
Couldn't get settingVersions event. Using empty settings.
Complete settings for atmcs.
Complete settings for atptg.
Complete settings for ataos.
Complete settings for atpneumatics.
Complete settings for athexapod.
Complete settings for atdome.
Complete settings for atdometrajectory.
Settings versions: {'atmcs': '                                                                                                                               ', 'atptg': '', 'ataos': 'current', 'atpneumatics': '                                                                                                                               ', 'athexapod': 'summit', 'atdome': 'test', 'atdometrajectory': ''}
[atmcs]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[atptg]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[ataos]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[atpneumatics]::[<State.STANDBY: 5>, <State.DISABLED: 

In [ ]:
# All components now enabled. - 17-Feb-2021 4:48 PM

In [10]:
# Ensure we're using Nasmyth 2
await atcs.rem.atptg.cmd_focusName.set_start(focus=3)

In [8]:
# Disable the dome
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.DISABLED, settingsToApply="test")
await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.DISABLED)

[<State.ENABLED: 2>, <State.DISABLED: 1>]

In [9]:
# Take event checking out the slew commands to test telescope only
atcs.check.atdome = False
atcs.check.atdometrajectory = False

In [11]:
# Run 1 bias as a test
await latiss.take_bias(1)

Generating group_id
BIAS 0001 - 0001
logMessage DDS read queue is filling: 36 of 100 elements


array([2021021700001])

In [12]:
# Move telescope to (az=180, el=80, rot=0). More convenient positioning for photos
current_az = 180.0
current_el = 80.0
current_rot = 0.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

Sending command
Stop tracking.
Unknown tracking state: 10
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
Got False
Telescope not in position
[Telescope] delta Alt = +000.001 deg; delta Az= +179.842 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +174.790 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +168.789 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +164.788 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +158.790 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +154.790 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = -000.000 deg; del

In [13]:
# Rotate to +80
current_az = 180.0
current_el = 80.0
current_rot = 80.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = +079.719 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = +074.408 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +068.409 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +062.407 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = +056.410 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = +050.407 deg 
[Telescope] delta Alt = +000.000

In [14]:
current_rot = 80.0
for i in range(14):
    final_delay = 30.0
    current_rot += 1.0
    await atcs.point_azel(current_az, current_el, rot_tel=current_rot)
    await asyncio.sleep(final_delay)
    await latiss.take_bias(1)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = +000.413 deg 
Got True
Waiting for telescope to settle.
[Telescope] delta Alt = +000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = -000.000 deg 
Axes in position.
Generating group_id
BIAS 0001 - 0001
Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; 

In [15]:
current_rot = 94.0
for i in range(6):
    final_delay = 30.0
    current_rot += 1.0
    await atcs.point_azel(current_az, current_el, rot_tel=current_rot)
    await asyncio.sleep(final_delay)
    await latiss.take_bias(1)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = +000.801 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = -000.000 deg 
Got True
Waiting for telescope to settle.
Axes in position.
Generating group_id
BIAS 0001 - 0001
Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; 

In [16]:
current_rot = 94.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)
await latiss.take_bias(1)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = -005.129 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = -001.499 deg 
Got True
Waiting for telescope to settle.
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +000.000 deg 
Axes in position.
Generating group_id
BIAS 0001 - 0001


array([2021021700022])

In [ ]:
# Holding wires away from ion pump controller (23)

In [17]:
await latiss.take_bias(1)

Generating group_id
BIAS 0001 - 0001
logMessage DDS read queue is filling: 19 of 100 elements


array([2021021700023])

In [ ]:
#  Centering gray connector (24)

In [18]:
await latiss.take_bias(1)

Generating group_id
BIAS 0001 - 0001


array([2021021700024])

In [19]:
# Move the rot back to home
current_rot = 0.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = -093.124 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = -087.307 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = -083.307 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = -077.309 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = -071.303 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = -065.313 deg 
[Telescope] delta Alt = +000.000

In [20]:
# Re-enable the dome
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [21]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.ENABLED, settingsToApply="test")

[<State.DISABLED: 1>, <State.ENABLED: 2>]

In [22]:
await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.ENABLED)

[<State.DISABLED: 1>, <State.ENABLED: 2>]

In [23]:
await atcs.prepare_for_flatfield()

Cover state <MirrorCoverState.CLOSED: 6>
Opening M1 cover.
Cover state <MirrorCoverState.CLOSED: 6>
Cover state <MirrorCoverState.CLOSED: 6>
Cover state <MirrorCoverState.INMOTION: 8>
Cover state <MirrorCoverState.OPENED: 7>
Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -041.000 deg; delta Az= +025.701 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -037.551 deg; delta Az= +025.131 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -031.549 deg; delta Az= +022.600 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -025.553 deg; delta Az= +017.968 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -019.64

In [26]:
# Take 50 biases seq # 26-75
await latiss.take_bias(50)

Generating group_id
BIAS 0001 - 0050
BIAS 0002 - 0050
BIAS 0003 - 0050
BIAS 0004 - 0050
BIAS 0005 - 0050
BIAS 0006 - 0050
BIAS 0007 - 0050
BIAS 0008 - 0050
BIAS 0009 - 0050
BIAS 0010 - 0050
BIAS 0011 - 0050
BIAS 0012 - 0050
BIAS 0013 - 0050
BIAS 0014 - 0050
BIAS 0015 - 0050
BIAS 0016 - 0050
BIAS 0017 - 0050
BIAS 0018 - 0050
BIAS 0019 - 0050
BIAS 0020 - 0050
BIAS 0021 - 0050
BIAS 0022 - 0050
logMessage DDS read queue is filling: 43 of 100 elements
BIAS 0023 - 0050
BIAS 0024 - 0050
BIAS 0025 - 0050
BIAS 0026 - 0050
logMessage DDS read queue is filling: 15 of 100 elements
BIAS 0027 - 0050
BIAS 0028 - 0050
logMessage DDS read queue is filling: 15 of 100 elements
BIAS 0029 - 0050
BIAS 0030 - 0050
BIAS 0031 - 0050
BIAS 0032 - 0050
BIAS 0033 - 0050
BIAS 0034 - 0050
BIAS 0035 - 0050
BIAS 0036 - 0050
BIAS 0037 - 0050
BIAS 0038 - 0050
BIAS 0039 - 0050
BIAS 0040 - 0050
BIAS 0041 - 0050
BIAS 0042 - 0050
BIAS 0043 - 0050
BIAS 0044 - 0050
BIAS 0045 - 0050
BIAS 0046 - 0050
BIAS 0047 - 0050
BIAS 0048 

array([2021021700026, 2021021700027, 2021021700028, 2021021700029,
       2021021700030, 2021021700031, 2021021700032, 2021021700033,
       2021021700034, 2021021700035, 2021021700036, 2021021700037,
       2021021700038, 2021021700039, 2021021700040, 2021021700041,
       2021021700042, 2021021700043, 2021021700044, 2021021700045,
       2021021700046, 2021021700047, 2021021700048, 2021021700049,
       2021021700050, 2021021700051, 2021021700052, 2021021700053,
       2021021700054, 2021021700055, 2021021700056, 2021021700057,
       2021021700058, 2021021700059, 2021021700060, 2021021700061,
       2021021700062, 2021021700063, 2021021700064, 2021021700065,
       2021021700066, 2021021700067, 2021021700068, 2021021700069,
       2021021700070, 2021021700071, 2021021700072, 2021021700073,
       2021021700074, 2021021700075])

In [28]:
# Take 10 10 second darks 76-85
await latiss.take_darks(10.0, 10)

Generating group_id
DARK 0001 - 0010
DARK 0002 - 0010
DARK 0003 - 0010
DARK 0004 - 0010
DARK 0005 - 0010
DARK 0006 - 0010
DARK 0007 - 0010
DARK 0008 - 0010
DARK 0009 - 0010
logMessage DDS read queue is filling: 14 of 100 elements
DARK 0010 - 0010
logMessage DDS read queue is filling: 15 of 100 elements


array([2021021700076, 2021021700077, 2021021700078, 2021021700079,
       2021021700080, 2021021700081, 2021021700082, 2021021700083,
       2021021700084, 2021021700085])

In [29]:
# Take 10 2 second flats 86-95
await latiss.take_flats(2.0, 10, filter='RG610', grating='empty_1')

Generating group_id
FLAT 0001 - 0010
FLAT 0002 - 0010
FLAT 0003 - 0010
FLAT 0004 - 0010
FLAT 0005 - 0010
FLAT 0006 - 0010
FLAT 0007 - 0010
FLAT 0008 - 0010
FLAT 0009 - 0010
FLAT 0010 - 0010


array([2021021700086, 2021021700087, 2021021700088, 2021021700089,
       2021021700090, 2021021700091, 2021021700092, 2021021700093,
       2021021700094, 2021021700095])

In [30]:
# Take flats for PTC 96-135
for i in range(20):
    exp = 0.2 * float(i+1)
    await latiss.take_flats(exp, 2, filter='empty_1', grating='empty_1')

Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
logMessage DDS read queue is filling: 12 of 100 elements
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 

In [33]:
await latiss.standby()

[atcamera]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atspectrograph]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atheaderservice]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atarchiver]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
All components in <State.STANDBY: 5>.


In [35]:
await atcs.enable()

Enabling all components
Gathering settings.
atdome check is disabled, skipping.
Couldn't get settingVersions event. Using empty settings.
Complete settings for atmcs.
Complete settings for atptg.
Complete settings for ataos.
Complete settings for atpneumatics.
Complete settings for athexapod.
Complete settings for atdometrajectory.
Settings versions: {'atmcs': '                                                                                                                               ', 'atptg': '', 'ataos': 'current', 'atpneumatics': '                                                                                                                               ', 'athexapod': 'summit', 'atdometrajectory': ''}
[atmcs]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[atptg]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[ataos]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[atpneumatics]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.E

In [36]:
await atcs.close_m1_vent()

M1 vent state <VentsPosition.CLOSED: 1>
M1 vents already closed.


In [37]:
await atcs.close_m1_cover()

Cover state <MirrorCoverState.OPENED: 7>
Closing M1 cover.
Sending command
Stop tracking.
Unknown tracking state: 10
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +030.984 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = -000.519 deg 
[Telescope] delta Alt = +029.725 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = +000.001 deg 
[Telescope] delta Alt = +028.780 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +024.646 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +021.096 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +017.214 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta 

In [40]:
await atcs.stop_tracking()

Stop tracking.


AckError: msg='Command failed', ackcmd=(ackcmd private_seqNum=2091141508, ack=<SalRetCode.CMD_FAILED: -302>, error=6612, result='Rejected : command not allowed in current state')

In [38]:
# Putting everything back in standby.
await atcs.standby()

Unable to transition atmcs to <State.STANDBY: 5> NoneType: None
.
Traceback (most recent call last):
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/csc_utils.py", line 148, in set_summary_state
    await cmd.start(timeout=timeout)
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/topics/remote_command.py", line 446, in start
    return await cmd_info.next_ackcmd(timeout=timeout)
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/topics/remote_command.py", line 183, in next_ackcmd
    raise base.AckError(msg="Command failed", ackcmd=ackcmd)
lsst.ts.salobj.base.AckError: msg='Command failed', ackcmd=(ackcmd private_seqNum=170175665, ack=<SalRetCode.CMD_NOPERM: -300>, error=0, result='ERROR: Command disable rejected while in TrackingEnabledState state.')

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/csc_utils.py", line 152, in set_summary_state
    ) from e
Runti

RuntimeError: Failed to transition ['atmcs'] to <State.STANDBY: 5>.

In [42]:
await salobj.set_summary_state(atcs.rem.atmcs, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [43]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.STANDBY, settingsToApply="test")

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [ ]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = False
atcs.check.atdometrajectory = False